# Deteccion y Lectura de Placas Vehiculares Colombianas con YOLOv8 y EasyOCR

**Estudiante:** Leydy Yohana Macareo Fuentes 
**Curso:** Inteligencia Artificial Avanzada - UNAB Digital 
**Dataset:** [Roboflow - placa-de-carro-sxy3a v4](https://universe.roboflow.com/cicatriz/placa-de-carro-sxy3a/dataset/4) 
**Basado en:** Cuaderno de referencia del profesor Alfredo Diaz (adaptado de YOLO11 a YOLOv8)

---

## Que hace este cuaderno?

Construye un sistema completo de **ALPR** (Automatic License Plate Recognition) para placas colombianas:

| Herramienta | Funcion |
|---|---|
| **Roboflow** | Descarga del dataset etiquetado de placas colombianas |
| **YOLOv8n** | Deteccion de la placa en la imagen (bounding box) |
| **OpenCV** | Recorte y preprocesamiento de la region de la placa |
| **EasyOCR** | Lectura de los caracteres de la placa |

## Flujo del sistema

```
Imagen -> YOLOv8 detecta la placa -> OpenCV recorta la region
       -> EasyOCR lee caracteres -> Extrae numero + ciudad
```

## Por que YOLOv8n y no YOLO11?

- **YOLOv8n** (nano) tiene solo 3.2M parametros vs 25.3M de YOLO11-Large
- Corre en **CPU** sin problemas, ideal para AWS EC2 sin GPU
- Menor consumo de memoria, cumple con la restriccion del profesor

## Requisitos previos
- Google Colab con GPU habilitada: **Runtime > Change runtime type > T4 GPU**
- Cuenta en [Roboflow](https://roboflow.com) (gratuita)

---
## Paso 1: Instalacion de Dependencias

Instalamos las librerias necesarias:

- **ultralytics**: Framework que incluye YOLOv8 para deteccion de objetos
- **roboflow**: SDK para descargar el dataset desde Roboflow
- **easyocr**: Motor OCR para leer texto en imagenes (soporta espanol e ingles)

In [ ]:
# Solo instalamos lo que NO viene preinstalado en Google Colab
!pip install ultralytics roboflow easyocr -q

---
## Paso 2: Importacion de Librerias

- **numpy**: Manipulacion de arrays numericos
- **pandas**: Tablas de datos (para las metricas)
- **matplotlib**: Graficas de entrenamiento
- **cv2 (OpenCV)**: Procesamiento de imagenes a nivel de pixel
- **easyocr**: Reconocimiento optico de caracteres
- **YOLO**: Clase principal de Ultralytics para cargar y entrenar modelos

In [ ]:
import numpy as np
import pandas as pd
import os
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import easyocr
from ultralytics import YOLO

print("Librerias importadas correctamente")

---
## Paso 3: Descarga del Dataset desde Roboflow

Usamos el dataset **placa-de-carro-sxy3a** del workspace **cicatriz** en Roboflow, version 4.

- **Clase:** `placa` (una sola clase)
- **Formato:** `yolov8` (compatible con YOLOv8)
- **Contenido:** Fotos de vehiculos colombianos con placas anotadas

> **Nota:** Reemplaza la API Key con la tuya propia. La puedes obtener en [app.roboflow.com](https://app.roboflow.com) > Settings > API Keys

In [ ]:
from roboflow import Roboflow

# Conectar con Roboflow
# IMPORTANTE: Reemplaza esta API Key con la tuya propia
rf = Roboflow(api_key="dSMfDD4uPaMCKEoGOP5q")

# Acceder al proyecto de deteccion de placas colombianas
project = rf.workspace("cicatriz").project("placa-de-carro-sxy3a")

# Seleccionar la version 4 del dataset
version = project.version(4)

# Descargar en formato YOLOv8 (no YOLO11)
dataset = version.download("yolov8", location="/content/placa-de-carro-4", overwrite=True)

print(f"\nDataset descargado en: {dataset.location}")

---
## Paso 4: Exploracion del Dataset

Antes de entrenar, verificamos la estructura del dataset:

```
placa-de-carro-4/
  train/
    images/   <-- Imagenes de entrenamiento
    labels/   <-- Anotaciones formato YOLO (.txt)
  valid/
    images/   <-- Imagenes de validacion
    labels/
  test/
    images/   <-- Imagenes de prueba
    labels/
  data.yaml   <-- Configuracion del dataset
```

El archivo `data.yaml` le dice a YOLO donde estan las imagenes y que clases existen.

In [ ]:
import yaml

# Ruta del dataset descargado
dataset_path = dataset.location
yaml_path = os.path.join(dataset_path, 'data.yaml')

# Leer y mostrar el archivo de configuracion
with open(yaml_path, 'r') as f:
    config = yaml.safe_load(f)

print("Contenido de data.yaml:")
print("-" * 40)
for key, value in config.items():
    print(f"  {key}: {value}")

# Contar imagenes por split
print("\nDistribucion del dataset:")
print("-" * 40)
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(dataset_path, split, 'images')
    if os.path.exists(img_dir):
        count = len(os.listdir(img_dir))
        print(f"  {split:10s}: {count:4d} imagenes")

print(f"\nClases: {config.get('names', [])}")

---
## Paso 5: Visualizacion de Muestras del Dataset

Mostramos algunas imagenes del dataset con sus bounding boxes para verificar que las anotaciones estan correctas.

**Formato de anotacion YOLO:** Cada linea del archivo `.txt` contiene:
```
clase  x_centro  y_centro  ancho  alto
```
Todos los valores estan **normalizados** entre 0 y 1.

In [ ]:
def mostrar_muestras_dataset(dataset_path, split='train', n=6):
    """Muestra imagenes del dataset con sus bounding boxes."""
    img_dir = os.path.join(dataset_path, split, 'images')
    lbl_dir = os.path.join(dataset_path, split, 'labels')
    imagenes = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
    imagenes = imagenes[:n]

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle(f'Muestras del dataset - Split: {split}', fontsize=16, fontweight='bold')

    for idx, (ax, img_file) in enumerate(zip(axes.flatten(), imagenes)):
        img_path = os.path.join(img_dir, img_file)
        lbl_path = os.path.join(lbl_dir, img_file.replace('.jpg', '.txt').replace('.png', '.txt').replace('.jpeg', '.txt'))

        img = Image.open(img_path)
        w, h = img.size
        ax.imshow(img)

        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cls, xc, yc, bw, bh = map(float, parts)
                        x1 = (xc - bw/2) * w
                        y1 = (yc - bh/2) * h
                        rect = patches.Rectangle((x1, y1), bw*w, bh*h,
                                                  linewidth=2, edgecolor='lime', facecolor='none')
                        ax.add_patch(rect)
                        ax.text(x1, y1-5, 'placa', color='lime', fontsize=9, fontweight='bold')

        ax.set_title(f'{img_file[:25]}...', fontsize=8)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

mostrar_muestras_dataset(dataset_path, split='train')

---
## Paso 6: Carga del Modelo YOLOv8

### Por que YOLOv8n (Nano)?

| Modelo | Parametros | Velocidad | Precision | Uso en AWS |
|--------|-----------|-----------|----------|------------|
| yolov8n | **3.2M** | Muy rapido | Buena | Ideal para CPU |
| yolov8s | 11.2M | Rapido | Mejor | OK en CPU |
| yolov8m | 25.9M | Medio | Muy buena | Necesita mas RAM |
| yolov8l | 43.7M | Lento | Excelente | Necesita GPU |

Usamos **yolov8n.pt** (Nano) porque:
- Solo 3.2M parametros = rapido en CPU
- Funciona bien en EC2 t2.large sin GPU
- Para una sola clase (placa) no necesitamos un modelo enorme
- El profesor indico que YOLO11 da problemas de memoria en AWS

In [ ]:
from ultralytics import YOLO

# Cargar YOLOv8 Nano preentrenado en COCO dataset (transfer learning)
# Se descarga automaticamente la primera vez (~6 MB)
model = YOLO("yolov8n.pt")

print("Modelo YOLOv8n cargado correctamente")
print("\nInformacion del modelo:")
model.info()

---
## Paso 7: Configuracion del Directorio del Dataset

Configuramos el directorio donde Ultralytics buscara el dataset. Esto es necesario para que las rutas relativas en `data.yaml` funcionen correctamente en Google Colab.

In [ ]:
from ultralytics import settings

# Configurar el directorio del dataset
settings.update({'datasets_dir': dataset_path})

print(f"Directorio del dataset configurado: {dataset_path}")
print(f"Archivo YAML: {yaml_path}")

---
## Paso 8: Entrenamiento del Modelo

### Transfer Learning

En lugar de entrenar desde cero, tomamos YOLOv8n **preentrenado en COCO** (80 clases generales) y lo **fine-tuneamos** con nuestras imagenes de placas colombianas. Esto nos da:
- Menos datos necesarios
- Convergencia mas rapida
- Mejor rendimiento final

### Parametros de entrenamiento

| Parametro | Valor | Descripcion |
|-----------|-------|-------------|
| `epochs` | 50 | Numero de pasadas completas sobre el dataset |
| `imgsz` | 640 | Tamano de imagen de entrada (640x640 pixeles) |
| `batch` | 16 | Imagenes procesadas simultaneamente |
| `patience` | 10 | Early stopping: para si no mejora en 10 epocas |
| `device` | 0 | Usar GPU 0 (T4 de Colab) |

> El entrenamiento toma aproximadamente 10-20 minutos con GPU T4.

In [ ]:
# Entrenamiento con fine-tuning
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,          # Early stopping
    name="detector_placas",
    project="/content/proyecto_placas",
    device=0,             # GPU
    verbose=True
)

print("\nEntrenamiento completado")
print(f"Resultados guardados en: /content/proyecto_placas/detector_placas")

---
## Paso 9: Analisis de Metricas de Entrenamiento

YOLO guarda metricas en `results.csv`. Analizamos:

- **train/cls_loss**: Perdida de clasificacion (que tan bien identifica la clase `placa`)
- **train/box_loss**: Perdida de localizacion (que tan preciso es el bounding box)
- **mAP50**: Mean Average Precision con IoU=0.5 (metrica principal)
- **Precision**: De las detecciones, cuantas son correctas
- **Recall**: De las placas reales, cuantas detecto

Un buen modelo deberia tener **mAP50 > 0.85** y ambas perdidas decreciendo.

In [ ]:
# Cargar metricas de entrenamiento
results_path = "/content/proyecto_placas/detector_placas/results.csv"
df_metrics = pd.read_csv(results_path)
df_metrics.columns = df_metrics.columns.str.strip()

print("Ultimas 5 epocas:")
print(df_metrics.tail(5).to_string(index=False))

# Graficar metricas
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Metricas de Entrenamiento - Detector de Placas Colombianas (YOLOv8n)',
             fontsize=14, fontweight='bold')

# Perdidas de entrenamiento
axes[0, 0].plot(df_metrics['epoch'], df_metrics['train/cls_loss'], 'r-', label='Cls Loss', linewidth=2)
axes[0, 0].plot(df_metrics['epoch'], df_metrics['train/box_loss'], 'b-', label='Box Loss', linewidth=2)
axes[0, 0].set_title('Perdidas de Entrenamiento')
axes[0, 0].set_xlabel('Epoca')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# mAP
axes[0, 1].plot(df_metrics['epoch'], df_metrics['metrics/mAP50(B)'], 'g-', label='mAP@50', linewidth=2)
axes[0, 1].plot(df_metrics['epoch'], df_metrics['metrics/mAP50-95(B)'], 'm-', label='mAP@50-95', linewidth=2)
axes[0, 1].set_title('Mean Average Precision (mAP)')
axes[0, 1].set_xlabel('Epoca')
axes[0, 1].set_ylabel('mAP')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Precision y Recall
axes[1, 0].plot(df_metrics['epoch'], df_metrics['metrics/precision(B)'], 'c-', label='Precision', linewidth=2)
axes[1, 0].plot(df_metrics['epoch'], df_metrics['metrics/recall(B)'], 'orange', label='Recall', linewidth=2)
axes[1, 0].set_title('Precision y Recall')
axes[1, 0].set_xlabel('Epoca')
axes[1, 0].set_ylabel('Valor')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Perdida de validacion
axes[1, 1].plot(df_metrics['epoch'], df_metrics['val/cls_loss'], 'r--', label='Val Cls Loss', linewidth=2)
axes[1, 1].plot(df_metrics['epoch'], df_metrics['val/box_loss'], 'b--', label='Val Box Loss', linewidth=2)
axes[1, 1].set_title('Perdidas de Validacion')
axes[1, 1].set_xlabel('Epoca')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/metricas_entrenamiento.png', dpi=150, bbox_inches='tight')
plt.show()

# Resumen final
mejor_epoch = df_metrics['metrics/mAP50(B)'].idxmax()
print(f"\nMejor epoca: {df_metrics.loc[mejor_epoch, 'epoch']:.0f}")
print(f"   mAP@50:     {df_metrics.loc[mejor_epoch, 'metrics/mAP50(B)']:.4f}")
print(f"   Precision:  {df_metrics.loc[mejor_epoch, 'metrics/precision(B)']:.4f}")
print(f"   Recall:     {df_metrics.loc[mejor_epoch, 'metrics/recall(B)']:.4f}")

---
## Paso 10: Evaluacion del Modelo en el Conjunto de Validacion

Evaluamos el mejor modelo (`best.pt`) con imagenes que nunca vio durante el entrenamiento. Esto nos da una estimacion realista del rendimiento.

Metricas clave:
- **mAP50**: Precision media con umbral IoU de 0.5 (estandar mas usado)
- **mAP50-95**: Mas estricto, promedio de IoU de 0.5 a 0.95

In [ ]:
# Cargar el mejor modelo entrenado
ruta_modelo = "/content/proyecto_placas/detector_placas/weights/best.pt"
modelo_entrenado = YOLO(ruta_modelo)

print(f"Modelo cargado desde: {ruta_modelo}")

# Evaluar en conjunto de validacion
print("\nEvaluando en conjunto de validacion...")
eval_results = modelo_entrenado.val(
    data=yaml_path,
    split='val'
)

print("\nResultados de evaluacion:")
print(f"   mAP@50:      {eval_results.box.map50:.4f}")
print(f"   mAP@50-95:   {eval_results.box.map:.4f}")
print(f"   Precision:   {eval_results.box.mp:.4f}")
print(f"   Recall:      {eval_results.box.mr:.4f}")

---
## Paso 11: Funciones de Preprocesamiento y OCR

Antes de leer los caracteres de la placa con EasyOCR, preprocesamos la imagen para mejorar la lectura:

1. **Escala de grises**: Simplifica la imagen
2. **Resize 2x**: Aumenta resolucion para mejor OCR
3. **CLAHE**: Mejora el contraste de forma adaptativa
4. **Filtro bilateral**: Reduce ruido preservando bordes de letras

Tambien incluimos una funcion para identificar la ciudad colombiana segun el prefijo de la placa.

In [ ]:
# Inicializar EasyOCR con soporte para espanol e ingles
print("Inicializando EasyOCR (puede tardar unos segundos la primera vez)...")
reader = easyocr.Reader(['es', 'en'], gpu=True)
print("EasyOCR inicializado")


def preprocesar_placa(imagen_placa):
    """
    Preprocesa la region de la placa para mejorar el OCR.
    1. Escala de grises
    2. Resize 2x
    3. CLAHE (mejora contraste)
    4. Filtro bilateral (reduce ruido)
    """
    gris = cv2.cvtColor(imagen_placa, cv2.COLOR_BGR2GRAY)

    # Aumentar resolucion 2x para mejor lectura OCR
    alto, ancho = gris.shape
    gris = cv2.resize(gris, (ancho * 2, alto * 2), interpolation=cv2.INTER_CUBIC)

    # CLAHE: mejora contraste de forma adaptativa
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gris = clahe.apply(gris)

    # Filtro bilateral: reduce ruido preservando bordes
    gris = cv2.bilateralFilter(gris, 11, 17, 17)

    return gris


def extraer_ciudad_placa(texto_placa):
    """
    Interpreta el texto OCR para extraer numero y ciudad de placas colombianas.
    Formato: ABC-123 (3 letras + 3 numeros)
    """
    import re

    # Prefijos de placas colombianas por ciudad/departamento
    prefijos_ciudades = {
        # Bogota D.C.
        'AAA': 'Bogota', 'BAA': 'Bogota', 'CAA': 'Bogota', 'DAA': 'Bogota',
        'EAA': 'Bogota', 'FAA': 'Bogota', 'GAA': 'Bogota', 'HAA': 'Bogota',
        # Antioquia (Medellin)
        'HIM': 'Medellin', 'MEP': 'Medellin', 'OCA': 'Medellin',
        # Valle del Cauca (Cali)
        'BXR': 'Cali', 'CXR': 'Cali', 'DXR': 'Cali',
        # Santander (Bucaramanga)
        'OBR': 'Bucaramanga', 'OBS': 'Bucaramanga', 'OBT': 'Bucaramanga',
        'OBU': 'Bucaramanga', 'OBV': 'Bucaramanga',
        # Atlantico (Barranquilla)
        'BCH': 'Barranquilla', 'BCJ': 'Barranquilla',
        # Cundinamarca
        'OHG': 'Cundinamarca', 'OHH': 'Cundinamarca',
    }

    # Limpiar texto: solo alfanumericos y guion
    texto_limpio = re.sub(r'[^A-Za-z0-9-]', '', texto_placa).upper()

    # Buscar patron de placa colombiana: 3 letras + 3 numeros
    patron = re.search(r'([A-Z]{3})[\-]?([0-9]{3})', texto_limpio)

    if patron:
        letras = patron.group(1)
        numeros = patron.group(2)
        placa_formateada = f"{letras}-{numeros}"

        # Buscar ciudad por prefijo
        ciudad = prefijos_ciudades.get(letras, None)
        if not ciudad:
            for prefijo, c in prefijos_ciudades.items():
                if letras[:2] == prefijo[:2]:
                    ciudad = c
                    break
        ciudad = ciudad or "Desconocida (ver tabla RUNT)"

        return placa_formateada, letras, numeros, ciudad

    return texto_limpio, "", "", "No identificada"


print("Funciones de procesamiento definidas")

---
## Paso 12: Pipeline Completo de Deteccion y Lectura

Esta funcion integra todo el flujo:
1. YOLOv8 detecta la placa en la imagen
2. OpenCV recorta la region detectada
3. Se preprocesa la imagen para mejor OCR
4. EasyOCR lee el texto de la placa
5. Se extrae numero y ciudad colombiana

In [ ]:
def detectar_y_leer_placa(ruta_imagen, modelo, reader_ocr, confianza_min=0.3, mostrar=True):
    """
    Pipeline completo: Detecta la placa con YOLOv8, recorta con OpenCV,
    lee con EasyOCR y extrae ciudad colombiana.
    """
    # 1. Cargar imagen
    imagen_bgr = cv2.imread(ruta_imagen)
    if imagen_bgr is None:
        print(f"No se pudo cargar la imagen: {ruta_imagen}")
        return None
    imagen_rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)
    alto, ancho = imagen_bgr.shape[:2]

    # 2. Deteccion con YOLOv8
    resultados_yolo = modelo.predict(
        source=ruta_imagen,
        conf=confianza_min,
        verbose=False
    )

    if len(resultados_yolo) == 0 or len(resultados_yolo[0].boxes) == 0:
        print("No se detectaron placas en la imagen.")
        return None

    # Tomar la deteccion con mayor confianza
    boxes = resultados_yolo[0].boxes
    mejor_idx = boxes.conf.argmax().item()
    x1, y1, x2, y2 = map(int, boxes.xyxy[mejor_idx].tolist())
    confianza = float(boxes.conf[mejor_idx])

    # 3. Recorte con margen
    margen = 5
    x1m = max(0, x1 - margen)
    y1m = max(0, y1 - margen)
    x2m = min(ancho, x2 + margen)
    y2m = min(alto, y2 + margen)

    recorte_bgr = imagen_bgr[y1m:y2m, x1m:x2m]
    recorte_rgb = imagen_rgb[y1m:y2m, x1m:x2m]

    # 4. Preprocesamiento para OCR
    placa_procesada = preprocesar_placa(recorte_bgr)

    # 5. OCR con EasyOCR
    ocr_resultados = reader_ocr.readtext(
        placa_procesada,
        detail=1,
        paragraph=False,
        allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-'
    )

    texto_raw = ' '.join([res[1] for res in ocr_resultados])
    conf_ocr = np.mean([res[2] for res in ocr_resultados]) if ocr_resultados else 0.0

    # 6. Extraer informacion de la placa colombiana
    placa_fmt, letras, numeros, ciudad = extraer_ciudad_placa(texto_raw)

    # 7. Visualizacion
    if mostrar:
        fig = plt.figure(figsize=(16, 6))
        fig.suptitle(f'Resultado: {placa_fmt} - {ciudad}',
                     fontsize=16, fontweight='bold', color='darkblue')

        # Imagen original con bounding box
        ax1 = fig.add_subplot(1, 3, 1)
        img_anotada = imagen_rgb.copy()
        cv2.rectangle(img_anotada, (x1, y1), (x2, y2), (0, 255, 0), 3)
        etiqueta = f"{placa_fmt} ({confianza:.0%})"
        cv2.putText(img_anotada, etiqueta, (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        ax1.imshow(img_anotada)
        ax1.set_title(f'Deteccion YOLOv8\nConfianza: {confianza:.1%}', fontsize=11)
        ax1.axis('off')

        # Recorte de la placa
        ax2 = fig.add_subplot(1, 3, 2)
        ax2.imshow(recorte_rgb)
        ax2.set_title('Recorte OpenCV\n(Region de la placa)', fontsize=11)
        ax2.axis('off')

        # Imagen preprocesada
        ax3 = fig.add_subplot(1, 3, 3)
        ax3.imshow(placa_procesada, cmap='gray')
        ax3.set_title(f'Preprocesada (OCR)\nTexto: "{texto_raw}"', fontsize=11)
        ax3.axis('off')

        plt.tight_layout()
        plt.show()

        print("\n" + "="*50)
        print("RESULTADO DEL ANALISIS")
        print("="*50)
        print(f"  Confianza YOLOv8:  {confianza:.1%}")
        print(f"  Texto OCR crudo:   '{texto_raw}'")
        print(f"  Placa detectada:   {placa_fmt}")
        print(f"  Letras:            {letras}")
        print(f"  Numeros:           {numeros}")
        print(f"  Ciudad/Depto:      {ciudad}")
        print(f"  Confianza OCR:     {conf_ocr:.1%}")
        print("="*50)

    return {
        'placa': placa_fmt,
        'letras': letras,
        'numeros': numeros,
        'ciudad': ciudad,
        'texto_raw': texto_raw,
        'confianza_yolo': confianza,
        'confianza_ocr': conf_ocr,
        'bbox': (x1, y1, x2, y2)
    }

print("Funcion detectar_y_leer_placa() lista")

---
## Paso 13: Prueba con Imagenes del Conjunto de Test

Probamos el pipeline completo con imagenes que el modelo nunca vio durante el entrenamiento.

In [ ]:
# Imagenes del conjunto de test
test_dir = os.path.join(dataset_path, 'test', 'images')

if os.path.exists(test_dir):
    imagenes_test = [
        os.path.join(test_dir, f)
        for f in os.listdir(test_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ][:5]

    print(f"Encontradas {len(imagenes_test)} imagenes de prueba")
    print("-" * 50)

    resultados_finales = []

    for i, ruta in enumerate(imagenes_test):
        print(f"\nProcesando imagen {i+1}/{len(imagenes_test)}: {os.path.basename(ruta)}")
        resultado = detectar_y_leer_placa(ruta, modelo_entrenado, reader, mostrar=True)
        if resultado:
            resultado['archivo'] = os.path.basename(ruta)
            resultados_finales.append(resultado)
else:
    print("No se encontro directorio de test. Usando validacion.")
    test_dir = os.path.join(dataset_path, 'valid', 'images')
    imagenes_test = [
        os.path.join(test_dir, f)
        for f in os.listdir(test_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ][:5]

---
## Paso 14: Tabla de Resultados

In [ ]:
if resultados_finales:
    df_resultados = pd.DataFrame(resultados_finales)
    cols_mostrar = ['archivo', 'placa', 'letras', 'numeros', 'ciudad', 'confianza_yolo', 'confianza_ocr']
    cols_disponibles = [c for c in cols_mostrar if c in df_resultados.columns]

    print("\nResumen de resultados:")
    print(df_resultados[cols_disponibles].to_string(index=False))

    print(f"\nPlacas detectadas exitosamente: {len(resultados_finales)}/{len(imagenes_test)}")
    print(f"Confianza promedio YOLOv8: {df_resultados['confianza_yolo'].mean():.1%}")
    if 'confianza_ocr' in df_resultados.columns:
        print(f"Confianza promedio OCR: {df_resultados['confianza_ocr'].mean():.1%}")
else:
    print("No se generaron resultados para mostrar.")

---
## Paso 15: Probar con una Imagen de Internet

Descargamos una imagen de prueba de internet para verificar que el modelo funciona con imagenes que no son del dataset.

In [ ]:
import requests
from io import BytesIO

# Imagen de prueba: placa colombiana desde Wikipedia
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/1a/Placa_de_autom%C3%B3vil_Colombia.jpg/640px-Placa_de_autom%C3%B3vil_Colombia.jpg"

try:
    response = requests.get(image_url)
    response.raise_for_status()

    img = Image.open(BytesIO(response.content))
    local_image_path = "/content/placa_prueba_internet.jpg"
    img.save(local_image_path)

    print(f"Imagen descargada: {local_image_path}")

    resultado_custom = detectar_y_leer_placa(
        local_image_path,
        modelo_entrenado,
        reader,
        confianza_min=0.25,
        mostrar=True
    )

except requests.exceptions.RequestException as e:
    print(f"Error al descargar la imagen: {e}")
except Exception as e:
    print(f"Error al procesar la imagen: {e}")

---
## Paso 16: Guardar y Descargar el Modelo Entrenado

Comprimimos los resultados del entrenamiento en un ZIP para descargar.

El archivo mas importante es **`best.pt`** dentro de `weights/` — es el modelo que subiremos a AWS para el backend.

In [ ]:
import shutil
from google.colab import files

# Verificar que el modelo existe
ruta_best = "/content/proyecto_placas/detector_placas/weights/best.pt"
if os.path.exists(ruta_best):
    tamano_mb = os.path.getsize(ruta_best) / (1024 * 1024)
    print(f"Modelo encontrado: best.pt ({tamano_mb:.1f} MB)")
else:
    print("Modelo no encontrado. Completaste el entrenamiento?")

# Comprimir la carpeta del proyecto
folder_path = "/content/proyecto_placas"
output_zip = "/content/leydy_placas_yolov8"

print("\nComprimiendo archivos del proyecto...")
shutil.make_archive(output_zip, 'zip', folder_path)
print(f"Archivo ZIP creado: {output_zip}.zip")

# Descargar el ZIP
print("\nIniciando descarga...")
files.download(f"{output_zip}.zip")

---
## Paso 17: Copiar best.pt a Google Drive (respaldo)

Guardamos una copia del modelo en Google Drive por si se desconecta Colab.

In [ ]:
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

# Crear carpeta de destino si no existe
destino = '/content/drive/MyDrive/UNAB/lector-placas'
os.makedirs(destino, exist_ok=True)

# Copiar best.pt
ruta_best = "/content/proyecto_placas/detector_placas/weights/best.pt"
if os.path.exists(ruta_best):
    shutil.copy2(ruta_best, os.path.join(destino, 'best.pt'))
    print(f"Modelo copiado a: {destino}/best.pt")
else:
    print("Modelo no encontrado. Ejecuta el entrenamiento primero.")

# Copiar tambien las metricas
ruta_metricas = "/content/metricas_entrenamiento.png"
if os.path.exists(ruta_metricas):
    shutil.copy2(ruta_metricas, os.path.join(destino, 'metricas_entrenamiento.png'))
    print(f"Metricas copiadas a: {destino}/metricas_entrenamiento.png")

---
## Resumen del Proyecto

### Que construimos?

Un sistema completo de **ALPR para placas colombianas** que:

1. Descarga un dataset etiquetado desde **Roboflow** (674 imagenes de entrenamiento)
2. Entrena **YOLOv8n** para detectar placas (clase: `placa`)
3. Usa **OpenCV** para recortar y preprocesar la region de la placa
4. Aplica **EasyOCR** para leer los caracteres
5. Identifica el **numero de placa** y la **ciudad/departamento** colombiano

### Siguiente paso

El archivo `best.pt` generado aqui se usara en el **backend FastAPI** que desplegaremos en **AWS EC2**, y sera consumido por la **app movil** con Expo/React Native.

---
**Leydy Yohana Macareo Fuentes** | UNAB Digital - IA Avanzada | Dataset: Roboflow - cicatriz/placa-de-carro-sxy3a v4